# Vector Database and Embeddings — Sport vs Cooking Article Classification

**Objectif.** Lire un article au format Microsoft Word (`.docx`) et déterminer si son contenu sémantique est plus proche du **sport** ou de la **cuisine**.

La solution reprend les concepts du cours : **embeddings avec Sentence-Transformers**, **normalisation des vecteurs**, **FAISS avec `IndexFlatL2`** et recherche sémantique par plus proches voisins. Le texte du document Word est d'abord tokenisé en ignorant la ponctuation avant la génération de l'embedding.

> Source utilisée pour les choix techniques : cours *Bases de données vectorielles* concernant les embeddings, la similarité, FAISS et `IndexFlatL2`.


## Table of contents

1. Environment and imports  
2. Configuration  
3. Reading a Word document  
4. Tokenization without punctuation  
5. Reference semantic database  
6. Embeddings with Sentence-Transformers  
7. FAISS vector index  
8. Classification functions  
9. Classification of one article  
10. Automated test suite  
11. Interpretation and limitations


## 1. Environment and imports

Ce notebook utilise les librairies nécessaires à la lecture de documents Word, à la génération d'embeddings et à la recherche vectorielle. Un modèle Sentence-Transformer multilingue est utilisé puisque les articles de test sont rédigés en français.

Lors de la première exécution, les dépendances manquantes sont installées automatiquement. Le modèle peut également être téléchargé s'il n'est pas déjà présent localement.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "docx": "python-docx",
    "pandas": "pandas",
    "numpy": "numpy",
}

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

from pathlib import Path
import re
import numpy as np
import pandas as pd
import faiss
from docx import Document
from sentence_transformers import SentenceTransformer


**Interpretation.** Les responsabilités sont séparées clairement : `python-docx` lit le document Word, `re` permet la tokenisation sans ponctuation, `SentenceTransformer` transforme le texte en vecteur sémantique dense et FAISS effectue la recherche de similarité entre vecteurs.


## 2. Configuration

Tous les chemins sont **relatifs** au notebook afin que le projet reste portable sur Moodle et GitHub.

Le modèle `paraphrase-multilingual-MiniLM-L12-v2` est un Sentence-Transformer multilingue adapté au français. Le cours utilise notamment `all-MiniLM-L6-v2` comme exemple ; ici, le choix d'un modèle multilingue permet de mieux correspondre à la langue des documents à classifier.


In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TEST_DIR = Path("tests")
TOP_REFERENCES_PER_CLASS = 3

SPORT = "sport"
COOKING = "cooking"


**Interpretation.** L'utilisation d'un modèle multilingue limite le risque que la langue française perturbe inutilement la représentation sémantique. Le nombre de références retenues par classe reste faible car la base vectorielle utilisée dans ce projet est volontairement petite.


## 3. Reading a Word document

La fonction suivante lit un fichier `.docx` et récupère tous ses paragraphes non vides.

Le nom du fichier n'est jamais utilisé pour effectuer la classification. Cela évite une fuite d'information : par exemple, un fichier nommé `test_sport_football.docx` ne doit pas être classé comme sport à cause de son nom, mais uniquement grâce au contenu de l'article.


In [ ]:
def read_docx(file_path: Path | str) -> str:
    """Read a Word document and return its non-empty paragraphs as one text."""
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Word document not found: {file_path}")

    if file_path.suffix.lower() != ".docx":
        raise ValueError("The input file must be a .docx document.")

    document = Document(file_path)
    paragraphs = [paragraph.text.strip() for paragraph in document.paragraphs if paragraph.text.strip()]

    if not paragraphs:
        raise ValueError("The Word document does not contain readable text.")

    return " ".join(paragraphs)


**Interpretation.** La validation du fichier d'entrée évite de produire un embedding dépourvu de sens. Si le document n'existe pas, n'est pas au format `.docx` ou ne contient aucun texte lisible, la fonction déclenche explicitement une erreur au lieu de retourner une classification non fiable.


## 4. Tokenization without punctuation

La consigne demande explicitement de tokeniser le document en ignorant la ponctuation, avec l'aide d'une IA générative.

Une IA générative a été utilisée pour proposer une méthode simple basée sur une expression régulière. La proposition a ensuite été vérifiée avec plusieurs assertions afin de contrôler son comportement.

**Prompt utilisé avec l'IA générative :**  
*« Écris une fonction Python pour tokeniser un texte français. Le texte doit être converti en minuscules, la ponctuation doit être ignorée, les caractères accentués et les chiffres doivent être conservés, et la fonction doit retourner une liste de tokens sans utiliser de librairie NLP. »*


In [ ]:
def tokenize_without_punctuation(text: str) -> list[str]:
    """Tokenize French text in lowercase while ignoring punctuation."""

    return re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", text.lower())


def prepare_text_for_embedding(text: str) -> str:
    """Tokenize text without punctuation and rebuild a clean string for the embedding model."""
    tokens = tokenize_without_punctuation(text)

    if not tokens:
        raise ValueError("No token remains after tokenization.")

    return " ".join(tokens)


**Interpretation.** La méthode de tokenisation reste simple et facilement vérifiable. La ponctuation est ignorée comme demandé, tandis que les lettres accentuées et les chiffres sont conservés. Les tokens sont ensuite réunis en une nouvelle chaîne de caractères avant d'être transmis au modèle d'embeddings.


### 4.1 Tokenizer verification

Avant d'utiliser cette fonction sur les articles, quelques cas simples sont vérifiés. Le test contrôle notamment la ponctuation, l'apostrophe, les majuscules, les caractères accentués et les chiffres.


In [ ]:
tokenizer_example = "L'équipe gagne 3-1 ! Très belle victoire, vraiment."
example_tokens = tokenize_without_punctuation(tokenizer_example)
print(example_tokens)

assert "équipe" in example_tokens
assert "victoire" in example_tokens
assert "!" not in example_tokens
assert "," not in example_tokens
assert example_tokens == ["l", "équipe", "gagne", "3", "1", "très", "belle", "victoire", "vraiment"]


**Interpretation.** Si toutes les assertions sont validées, la tokenisation respecte le comportement attendu : la ponctuation ne devient pas un token, les majuscules sont normalisées en minuscules et les mots contenant des accents sont conservés.


## 5. Reference semantic database

Pour pouvoir décider si un article appartient au domaine du sport ou de la cuisine, il faut disposer de références représentant sémantiquement les deux catégories.

Au lieu de définir chaque catégorie avec un seul mot-clé, plusieurs phrases sont utilisées pour représenter différents contextes liés au **sport** et à la **cuisine**. Cette approche limite la dépendance à un mot précis et exploite davantage la proximité sémantique offerte par les embeddings.


In [ ]:
REFERENCE_DATA = [

    {"label": SPORT, "text": "Une équipe dispute un match de football, marque des buts et défend son score."},
    {"label": SPORT, "text": "Le joueur de tennis sert, échange plusieurs balles et remporte le set."},
    {"label": SPORT, "text": "Le cycliste attaque dans la montée et franchit la ligne d'arrivée devant le peloton."},
    {"label": SPORT, "text": "Une équipe de basket défend, récupère le ballon et inscrit un panier à trois points."},
    {"label": SPORT, "text": "La coureuse prépare une compétition avec des séances d'entraînement et un objectif de chrono."},
    {"label": SPORT, "text": "Le nageur améliore sa technique, son endurance et son temps sur la distance."},
    {"label": SPORT, "text": "L'entraîneur modifie la tactique et effectue un remplacement pendant la rencontre."},
    {"label": SPORT, "text": "Le championnat se joue sur plusieurs journées avec un classement, des victoires et des défaites."},
    {"label": SPORT, "text": "L'athlète s'échauffe avant l'épreuve puis accélère dans la dernière partie de la course."},
    {"label": SPORT, "text": "Le gardien arrête un tir décisif et permet à son équipe de conserver l'avantage."},


    {"label": COOKING, "text": "Une recette mélange des ingrédients avant une cuisson au four."},
    {"label": COOKING, "text": "Le chef fait revenir des légumes dans une poêle puis ajoute une sauce et des épices."},
    {"label": COOKING, "text": "Pour préparer un gâteau, on mélange farine, œufs, sucre et beurre avant d'enfourner."},
    {"label": COOKING, "text": "Les pâtes cuisent dans l'eau salée puis sont mélangées avec une sauce et du fromage."},
    {"label": COOKING, "text": "Une soupe mijote dans une casserole avec des légumes, du bouillon et des herbes."},
    {"label": COOKING, "text": "Le cuisinier assaisonne le plat avec du sel, du poivre et des aromates."},
    {"label": COOKING, "text": "Le restaurant propose un menu avec une entrée, un plat principal et un dessert."},
    {"label": COOKING, "text": "La pâtisserie demande une pâte précise, une température de cuisson et un temps de repos."},
    {"label": COOKING, "text": "Les légumes sont épluchés, découpés puis cuits avant le dressage de l'assiette."},
    {"label": COOKING, "text": "Une sauce est réduite doucement pour concentrer les saveurs avant de servir le plat."},
]

reference_df = pd.DataFrame(REFERENCE_DATA)
reference_df


**Interpretation.** Les deux classes contiennent chacune dix références, ce qui évite de favoriser une catégorie uniquement parce qu'elle possède davantage de vecteurs. Les exemples couvrent plusieurs sous-domaines : football, tennis, cyclisme ou natation pour le sport ; pâtisserie, pâtes, soupe ou restauration pour la cuisine.


## 6. Embeddings with Sentence-Transformers

Chaque texte de référence est d'abord nettoyé avec la même méthode de tokenisation que celle utilisée pour les futurs articles. Il est ensuite transformé en embedding dense avec Sentence-Transformers.

Les embeddings sont normalisés afin que leur norme soit égale à 1. Le cours indique que, pour des vecteurs normalisés, minimiser la distance L2 revient à maximiser la similarité cosinus.

La relation est :

\[
$d^2$ = 2 - ( 2 \ $\cos$($\theta$) )
\]

Ainsi, plus deux vecteurs normalisés sont proches selon la distance L2, plus leurs contenus sont proches sémantiquement.


In [ ]:
model = SentenceTransformer(MODEL_NAME)

clean_reference_texts = [
    prepare_text_for_embedding(text)
    for text in reference_df["text"].tolist()
]

reference_embeddings = model.encode(
    clean_reference_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype("float32")

print("Embedding matrix shape:", reference_embeddings.shape)
print("First vector norm:", np.linalg.norm(reference_embeddings[0]))


**Interpretation.** Tous les textes sont transformés en vecteurs de même dimension, ce qui est indispensable pour les stocker dans FAISS. Une norme proche de 1 confirme que la normalisation a bien été appliquée.


## 7. FAISS vector index

Le cours présente `IndexFlatL2` comme un index FAISS simple utilisant une recherche exhaustive avec la distance L2.

Cette solution est adaptée ici car la base contient seulement vingt vecteurs de référence. Il n'est donc pas nécessaire d'utiliser une méthode de recherche approximative plus complexe.


In [ ]:
embedding_dimension = reference_embeddings.shape[1]
vector_index = faiss.IndexFlatL2(embedding_dimension)
vector_index.add(reference_embeddings)

print("FAISS dimension:", vector_index.d)
print("Stored vectors:", vector_index.ntotal)


**Interpretation.** `IndexFlatL2` compare le vecteur de la requête à tous les vecteurs stockés. Avec seulement vingt références, cette recherche exacte reste très rapide et permet de privilégier la simplicité et la précision.


## 8. Classification functions

Pour classifier un texte, son embedding est comparé à tous les embeddings stockés dans FAISS.

Pour chaque catégorie, le système sélectionne les trois références possédant la plus grande similarité cosinus, puis calcule leur similarité moyenne. La catégorie ayant le score moyen le plus élevé devient la prédiction.

Cette stratégie utilise plusieurs voisins au lieu de se fier uniquement au voisin le plus proche.


In [ ]:
def squared_l2_to_cosine(squared_l2_distance: float) -> float:
    """Convert squared L2 distance between normalized vectors to cosine similarity."""
    return 1.0 - (squared_l2_distance / 2.0)


def classify_text(text: str, top_per_class: int = TOP_REFERENCES_PER_CLASS) -> dict:
    """Classify a text as sport or cooking using normalized embeddings and FAISS."""
    clean_text = prepare_text_for_embedding(text)

    query_embedding = model.encode(
        [clean_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    distances, indices = vector_index.search(query_embedding, vector_index.ntotal)

    neighbour_rows = []
    for distance, reference_index in zip(distances[0], indices[0]):
        row = reference_df.iloc[int(reference_index)]
        neighbour_rows.append({
            "label": row["label"],
            "reference": row["text"],
            "squared_l2": float(distance),
            "cosine_similarity": squared_l2_to_cosine(float(distance)),
        })

    neighbours_df = pd.DataFrame(neighbour_rows)

    class_scores = (
        neighbours_df
        .sort_values("cosine_similarity", ascending=False)
        .groupby("label", group_keys=False)
        .head(top_per_class)
        .groupby("label")["cosine_similarity"]
        .mean()
        .sort_values(ascending=False)
    )

    prediction = class_scores.index[0]
    confidence_gap = float(class_scores.iloc[0] - class_scores.iloc[1])

    return {
        "prediction": prediction,
        "scores": class_scores.to_dict(),
        "confidence_gap": confidence_gap,
        "nearest_references": neighbours_df.head(6),
        "clean_text": clean_text,
    }


def classify_docx(file_path: Path | str) -> dict:
    """Read a Word article and classify it as sport or cooking."""
    article_text = read_docx(file_path)
    result = classify_text(article_text)
    result["file"] = str(file_path)
    return result


**Interpretation.** Calculer la moyenne de plusieurs références proches rend la décision moins dépendante d'une seule phrase de référence. Le `confidence_gap` mesure l'écart entre les scores des deux classes. Il ne s'agit pas d'une probabilité : une faible valeur indique simplement que les deux catégories obtiennent des scores proches et que le résultat doit être interprété avec prudence.


## 9. Classification of one article

La cellule suivante montre le fonctionnement complet du programme sur un document Word.

Pour tester un autre fichier, il suffit de modifier `ARTICLE_PATH` afin qu'il corresponde à un autre document `.docx` présent dans le dossier du projet.


In [ ]:
ARTICLE_PATH = TEST_DIR / "test_sport_football.docx"

single_result = classify_docx(ARTICLE_PATH)

print("File:", single_result["file"])
print("Prediction:", single_result["prediction"])
print("Class scores:", single_result["scores"])
print("Confidence gap:", round(single_result["confidence_gap"], 4))

display(single_result["nearest_references"][["label", "cosine_similarity", "reference"]])


**Interpretation.** La prédiction finale ne doit pas être examinée seule. Les références récupérées par FAISS permettent également d'observer quels textes sont sémantiquement les plus proches de l'article. Cela fournit une forme simple d'explicabilité du résultat.


## 10. Automated test suite

Les tests sont organisés en trois groupes :

- quatre articles clairement liés au sport ;
- quatre articles clairement liés à la cuisine ;
- deux cas plus ambigus contenant du vocabulaire pouvant appartenir aux deux domaines.

La classe attendue est conservée dans le notebook et n'est pas écrite dans le contenu des documents utilisés pour la classification.


In [ ]:
TEST_CASES = {
    "test_sport_football.docx": SPORT,
    "test_sport_tennis.docx": SPORT,
    "test_sport_cycling.docx": SPORT,
    "test_sport_basketball.docx": SPORT,
    "test_cuisine_pasta.docx": COOKING,
    "test_cuisine_pastry.docx": COOKING,
    "test_cuisine_soup.docx": COOKING,
    "test_cuisine_restaurant.docx": COOKING,
    "test_boundary_sport_nutrition.docx": SPORT,
    "test_boundary_cuisine_competition.docx": COOKING,
}

test_results = []

for filename, expected_label in TEST_CASES.items():
    result = classify_docx(TEST_DIR / filename)
    test_results.append({
        "file": filename,
        "expected": expected_label,
        "predicted": result["prediction"],
        "correct": result["prediction"] == expected_label,
        "sport_score": result["scores"].get(SPORT, np.nan),
        "cooking_score": result["scores"].get(COOKING, np.nan),
        "confidence_gap": result["confidence_gap"],
    })

results_df = pd.DataFrame(test_results)
accuracy = results_df["correct"].mean()

print(f"Accuracy on provided tests: {accuracy:.1%}")
display(results_df)

confusion_matrix = pd.crosstab(
    results_df["expected"],
    results_df["predicted"],
    rownames=["Expected"],
    colnames=["Predicted"],
)

display(confusion_matrix)


**Interpretation.** L'accuracy globale doit être analysée avec les résultats individuels, en particulier les cas ambigus et les valeurs de `confidence_gap`.

Une bonne performance sur ces dix documents montre que le système fonctionne sur les tests prévus, mais elle ne prouve pas qu'il généralisera correctement à n'importe quel article réel. Un cas possédant un faible `confidence_gap` est particulièrement intéressant à analyser manuellement.


## 11. Interpretation and limitations

Cette solution est un **classifieur sémantique par plus proches voisins**. Il ne s'agit pas d'un modèle supervisé entraîné sur un grand dataset étiqueté.

Son principal avantage est sa simplicité : la décision peut être reliée aux textes de référence retrouvés par FAISS. Le projet permet donc d'appliquer directement plusieurs notions du cours : embeddings, représentation vectorielle, normalisation, distance L2 et recherche de similarité dans une base vectorielle.

Plusieurs limites doivent cependant être prises en compte :

- la base de références est petite et définie manuellement ;
- le modèle d'embeddings peut reproduire certains biais présents dans ses données d'entraînement ;
- la suppression de la ponctuation est imposée par la consigne, mais peut supprimer une partie du contexte ;
- les scores de similarité obtenus ne sont pas des probabilités ;
- certains articles peuvent réellement mélanger sport et cuisine ;
- l'utilisation d'une IA générative pour aider à produire les références et les tests peut rendre les documents stylistiquement proches et donc faciliter artificiellement la classification.

Pour obtenir une évaluation plus robuste, il serait pertinent d'ajouter des articles réels ou rédigés indépendamment, qui n'ont pas été produits en même temps que les textes de référence.
